In [1]:
import os
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

In [ ]:
SYNTHETIC_SAMPLES = 10000
METADATA_FILE = "spoofing_metadata.json"


In [ ]:
from gps_ids import schema

_schema = schema.get("2d")
X, y = schema.load_real(_schema)
df = X.copy()
df[_schema.label_column] = y


In [4]:
# Load or create metadata
def _columns_of(meta):
    d = meta.to_dict()
    table = list(d['tables'].values())[0] if 'tables' in d else d
    return set(table['columns'])


def _detect(frame):
    meta = Metadata()
    meta.detect_table_from_dataframe(data=frame, table_name='spoofing_data')
    return meta


metadata = None
if os.path.exists(METADATA_FILE):
    cached = Metadata.load_from_json(METADATA_FILE)
    if _columns_of(cached) == set(df.columns):
        metadata = cached
        print(f"Loaded metadata from {METADATA_FILE}")
    else:
        print(f"{METADATA_FILE} does not describe this dataframe; re-detecting.")

if metadata is None:
    metadata = _detect(df)
    metadata.save_to_json(METADATA_FILE, mode='overwrite')
    print(f"Created and saved new metadata to {METADATA_FILE}")

Loaded metadata from spoofing_metadata.json


In [5]:
# Synthesizer configurations
# If you don't have a GPU available, it will automatically use CPU.
synthesizers = [
    {
        'name': 'CTGAN',
        'class': CTGANSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'epochs': 4000,
            'verbose': True,
            'cuda': True
        },
        'output': "../../../Datasets/excel/2dSpoofingDataset/spoofing-CTGAN-synthetic.csv",
        'pickle': "spoofing-CTGAN-synthesizer.pkl"
    },
    {
        'name': 'GaussianCopula',
        'class': GaussianCopulaSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
        },
        'output': "../../../Datasets/excel/2dSpoofingDataset/spoofing-GaussianCopula-synthetic.csv",
        'pickle': "spoofing-GaussianCopula-synthesizer.pkl"
    },
    {
        'name': 'TVAE',
        'class': TVAESynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'epochs': 2000,
            'verbose': True,
            'cuda': True
        },
        'output': "../../../Datasets/excel/2dSpoofingDataset/spoofing-TVAE-synthetic.csv",
        'pickle': "spoofing-TVAE-synthesizer.pkl"
    }
]

In [6]:
for synth in synthesizers:
    print(f"\nProcessing {synth['name']}...")
    SynthClass = synth['class']
    synthesizer = SynthClass(metadata, **synth['params'])
    synthesizer.fit(df)
    synthesizer.reset_sampling()
    synthetic_data = synthesizer.sample(num_rows=SYNTHETIC_SAMPLES)
    print(f"Synthetic data preview for {synth['name']}:")
    print(synthetic_data.head())
    synthetic_data.to_csv(synth['output'], index=False)
    print(f"Synthetic dataset saved to {synth['output']}")



    synthesizer.save(filepath=synth['pickle'])
    print(f"Trained synthesizer saved to {synth['pickle']}")


Processing CTGAN...


Gen. (0.07) | Discrim. (-0.13): 100%|██████████| 300/300 [3:54:30<00:00, 46.90s/it]   


Synthetic data preview for CTGAN:
   PRN           DO            PD             RX            TOW  \
0   26 -1469.009293 -3.106631e+06  173642.234665  173640.008000   
1   26 -1580.064075  2.428561e+07  174191.324077  174136.296894   
2    3 -2562.311005  2.728081e+07  174145.436196  174154.553520   
3   11  3146.989769  2.546092e+07  491612.961294  491804.687865   
4    8  4897.340735  4.904339e+06  263296.665881  263284.321051   

              CP             EC             LC             PC            PIP  \
0   36835.543747  134046.410291  129298.441208  163438.081043  133840.726481   
1  298054.144042  142367.853539  145993.150936  168250.741071 -172252.361833   
2  159472.795693   34240.257966   32613.390660   41879.739993  -51358.394538   
3 -493057.233060  115210.384700  103608.278806  132583.649164   89763.577745   
4 -676871.055535   56341.405548   58776.331424   70527.638020   71543.268704   

            PQP          TCD        CN0  Output  
0   3016.421784 -1496.005442  47

Loss: -49.124: 100%|██████████| 300/300 [1:39:26<00:00, 19.89s/it]


Synthetic data preview for TVAE:
   PRN           DO            PD             RX            TOW  \
0    8  4393.815706  4.338496e+06  263616.842504  263614.759434   
1   16  -875.829988  2.652510e+06  263916.850625  263907.897408   
2   16  -468.825679  2.624197e+06  262808.568246  262840.751828   
3    7  3679.449039  3.955002e+06  263542.709998  263575.283089   
4    7  1600.225893  2.346849e+06  263794.467163  263790.999863   

              CP             EC             LC             PC            PIP  \
0 -270947.779496  132033.835722  138558.590865  148476.079568  135110.623794   
1  118174.082714  111280.480933  118341.795566  122560.712814 -125384.367804   
2   52810.369912   90269.661175   93832.592297  107979.291372  -99726.704320   
3 -325468.985518  141775.007908  144072.007665  136200.045608  143587.580353   
4 -281868.323875  133043.904954  129701.895869  135197.250560 -137150.675521   

             PQP          TCD        CN0  Output  
0   -2528.030467  4620.924661  4